# ITRI625 – Phishing E-mail Detection with a BiLSTM
**Student:** _your name / student number_  &nbsp;|&nbsp; **Module:** ITRI625 &nbsp;|&nbsp; **Dataset:** Phishing Email Dataset (Kaggle)

| # | Section | Rubric item |
|---|---|---|
| 1 | Setup & configuration | – |
| 2 | Data loading, cleaning & EDA | Dataset preparation |
| 3 | Train / validation / test split | Dataset divided into sets (5) |
| 4 | Baseline: TF-IDF + Logistic Regression | Model development |
| 5 | Preprocessing & BiLSTM model | Model development (20) |
| 6 | Training loop: per-epoch logging + early stopping | Metric logging (15), Patience (5) |
| 7 | Metric graphs | Metric graphs (15) |
| 8 | Final test evaluation | Model result analysis |
| 9 | Save model artefacts | Desktop application (15) |
| 10 | Explainable AI (LIME) | Additional features (10) |

> **How to run:** put `Phishing_Email.csv` next to this notebook → *Kernel ▸ Restart & Run All* → save the notebook **with outputs** (a missing-output deduction is −10).

## 1. Setup & configuration

In [ ]:
import json, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             classification_report, log_loss)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)

# ---- configuration (change here, nowhere else) ----
DATA_PATH      = "Phishing_Email.csv"
TEXT_COL       = "Email Text"
LABEL_COL      = "Email Type"
POSITIVE_LABEL = "Phishing Email"      # the class we want to detect

MAX_LEN, MAX_VOCAB, MIN_FREQ = 256, 30_000, 2
EMBED_DIM, HIDDEN, DROPOUT   = 128, 128, 0.4
BATCH, LR, WEIGHT_DECAY      = 64, 1e-3, 1e-5
MAX_EPOCHS, PATIENCE, MIN_DELTA = 30, 4, 1e-4
THRESHOLD = 0.5

print("Device:", DEVICE)

**Shared module.** The cell below writes `phishing_core.py` (tokeniser, vocabulary, BiLSTM class, inference wrapper). The notebook, the FastAPI service and the desktop app all import this *same* file, so training and serving behave identically.

In [ ]:
%%writefile phishing_core.py
"""Shared preprocessing + model code for the ITRI625 phishing e-mail detector.

This file is written by the training notebook (%%writefile) and imported by both
the notebook and the FastAPI service, so training and serving can never drift apart.
"""
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

PAD, UNK = "<pad>", "<unk>"
PAD_ID, UNK_ID = 0, 1

_URL = re.compile(r"https?://\S+|www\.\S+")
_EMAIL = re.compile(r"\S+@\S+\.\S+")
_TOKEN = re.compile(r"<url>|<email>|[a-z]+(?:'[a-z]+)?|\d+|[!$%]")


# --------------------------------------------------------------------------- #
# Text preprocessing
# --------------------------------------------------------------------------- #
def clean_text(text: str) -> str:
    """Lower-case and replace URLs / e-mail addresses with placeholder tokens."""
    text = str(text).lower()
    text = _URL.sub(" <url> ", text)
    text = _EMAIL.sub(" <email> ", text)
    return text


def tokenize(text: str) -> list:
    return _TOKEN.findall(clean_text(text))


def build_vocab(token_lists, max_size=30000, min_freq=2) -> dict:
    """Build the vocabulary from TRAINING tokens only (prevents leakage)."""
    counts = Counter(t for toks in token_lists for t in toks)
    itos = [PAD, UNK] + [w for w, c in counts.most_common(max_size - 2) if c >= min_freq]
    return {w: i for i, w in enumerate(itos)}


def encode_batch(texts, vocab: dict, max_len: int):
    """Convert raw texts to (padded id tensor, true-length tensor)."""
    ids = np.zeros((len(texts), max_len), dtype=np.int64)
    lengths = np.ones(len(texts), dtype=np.int64)
    for i, text in enumerate(texts):
        toks = tokenize(text)[:max_len]
        seq = [vocab.get(t, UNK_ID) for t in toks] or [UNK_ID]
        ids[i, : len(seq)] = seq
        lengths[i] = len(seq)
    return torch.from_numpy(ids), torch.from_numpy(lengths)


# --------------------------------------------------------------------------- #
# Model
# --------------------------------------------------------------------------- #
class PhishingBiLSTM(nn.Module):
    """Embedding -> BiLSTM -> [mean-pool ; max-pool] -> Dense(ReLU) -> logit."""

    def __init__(self, vocab_size, embed_dim=128, hidden=128, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden * 4, 64)  # mean(2h) + max(2h)
        self.act = nn.ReLU()
        self.out = nn.Linear(64, 1)

    def forward(self, ids, lengths):
        emb = self.drop(self.embedding(ids))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=ids.size(1))

        steps = torch.arange(ids.size(1), device=ids.device)[None, :]
        mask = (steps < lengths[:, None]).unsqueeze(-1)  # (B, T, 1)
        mean = (out * mask).sum(1) / lengths.unsqueeze(1).float()
        mx = out.masked_fill(~mask, -1e9).max(1).values

        h = self.drop(torch.cat([mean, mx], dim=1))
        h = self.drop(self.act(self.fc1(h)))
        return self.out(h).squeeze(1)  # raw logit


# --------------------------------------------------------------------------- #
# Inference wrapper (used by the API and the XAI section of the notebook)
# --------------------------------------------------------------------------- #
class PhishingClassifier:
    def __init__(self, model, vocab, config, device="cpu"):
        self.model = model.to(device).eval()
        self.vocab = vocab
        self.config = config
        self.device = device
        self.threshold = float(config.get("threshold", 0.5))

    @classmethod
    def from_dir(cls, directory="artifacts", device=None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        d = Path(directory)
        config = json.loads((d / "config.json").read_text())
        vocab = json.loads((d / "vocab.json").read_text())
        model = PhishingBiLSTM(len(vocab), config["embed_dim"], config["hidden"], config["dropout"])
        model.load_state_dict(torch.load(d / "model.pt", map_location=device))
        return cls(model, vocab, config, device)

    @torch.no_grad()
    def predict_proba(self, texts, batch_size=64) -> np.ndarray:
        """Return P(phishing) for each text."""
        if isinstance(texts, str):
            texts = [texts]
        probs = []
        for i in range(0, len(texts), batch_size):
            ids, lengths = encode_batch(texts[i : i + batch_size], self.vocab, self.config["max_len"])
            logits = self.model(ids.to(self.device), lengths.to(self.device))
            probs.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(probs)

    def predict(self, text: str) -> dict:
        p = float(self.predict_proba([text])[0])
        is_phish = p >= self.threshold
        return {
            "label": "phishing" if is_phish else "legitimate",
            "is_phishing": bool(is_phish),
            "probability": p,
            "threshold": self.threshold,
        }

    # -- Explainable AI (LIME) ------------------------------------------------ #
    def lime_explanation(self, text: str, num_features=10, num_samples=500):
        from lime.lime_text import LimeTextExplainer

        explainer = LimeTextExplainer(class_names=["legitimate", "phishing"], random_state=42)

        def predict_fn(batch):
            p = self.predict_proba(list(batch))
            return np.column_stack([1.0 - p, p])

        return explainer.explain_instance(
            text, predict_fn, num_features=num_features, num_samples=num_samples
        )

    def explain(self, text: str, num_features=10, num_samples=500) -> list:
        """Return [{word, weight}] - positive weight pushes towards 'phishing'."""
        exp = self.lime_explanation(text, num_features, num_samples)
        return [{"word": w, "weight": float(s)} for w, s in exp.as_list()]

In [ ]:
from phishing_core import (tokenize, build_vocab, encode_batch, PhishingBiLSTM, PhishingClassifier)

## 2. Data loading, cleaning & EDA
Cleaning steps (each one is counted so it can be reported in the write-up): drop missing/empty texts, remove texts that appear with **conflicting labels**, then remove exact duplicates. De-duplicating *before* splitting prevents the same e-mail landing in both train and test (data leakage).

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape, "| columns:", list(raw.columns))
print("\nRaw label values:\n", raw[LABEL_COL].value_counts(dropna=False))
assert POSITIVE_LABEL in raw[LABEL_COL].unique(), "POSITIVE_LABEL not found - check the label values above"
raw.head()

In [ ]:
df = raw[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label_raw"})
df["label"] = (df["label_raw"] == POSITIVE_LABEL).astype(int)
log = {"raw rows": len(df)}

df = df.dropna(subset=["text"]);                         log["after dropping NaN"] = len(df)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0];                        log["after dropping empty"] = len(df)

conflict = df.groupby("text")["label"].nunique()
df = df[~df["text"].isin(conflict[conflict > 1].index)];  log["after removing conflicting labels"] = len(df)
df = df.drop_duplicates(subset="text").reset_index(drop=True); log["after removing duplicates"] = len(df)

pd.Series(log, name="rows").to_frame()

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

counts = df["label"].value_counts().sort_index()
bars = ax[0].bar(["Legitimate", "Phishing"], counts.values, color=["#4c9f70", "#c0392b"])
ax[0].bar_label(bars, fmt="%d"); ax[0].set_title("Class distribution"); ax[0].set_ylabel("e-mails")

for lab, name, col in [(0, "Legitimate", "#4c9f70"), (1, "Phishing", "#c0392b")]:
    ax[1].hist(df.loc[df.label == lab, "n_words"].clip(upper=1500), bins=50, alpha=0.6, label=name, color=col)
ax[1].set_title("E-mail length (words, clipped at 1500)"); ax[1].set_xlabel("words"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"Phishing share: {df.label.mean():.1%}")
print(df.groupby("label")["n_words"].describe().round(1))

## 3. Train / validation / test split (70 / 15 / 15)
Stratified so every split keeps the same class ratio. The **test set is only used once, at the very end** (section 8); the validation set drives early stopping.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "e-mails": [len(train_df), len(val_df), len(test_df)],
    "phishing share": [d.label.mean() for d in (train_df, val_df, test_df)],
}).set_index("split")
summary["share of data"] = summary["e-mails"] / len(df)
summary.round(3)

## 4. Baseline: TF-IDF + Logistic Regression
A strong classical baseline. If the BiLSTM cannot beat it, the extra complexity is not justified – so it gives the results analysis something honest to compare against. The vectoriser is fitted on the **training set only**.

In [ ]:
def compute_metrics(y_true, y_prob, loss=None, thr=THRESHOLD):
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    m = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "auc":       roc_auc_score(y_true, y_prob),
    }
    if loss is not None:
        m["loss"] = loss
    return m

tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words="english")
X_tr = tfidf.fit_transform(train_df["text"]); X_va = tfidf.transform(val_df["text"]); X_te = tfidf.transform(test_df["text"])
lr = LogisticRegression(max_iter=1000, C=4.0).fit(X_tr, train_df["label"])

p_va_lr, p_te_lr = lr.predict_proba(X_va)[:, 1], lr.predict_proba(X_te)[:, 1]
base_val = compute_metrics(val_df["label"], p_va_lr, loss=log_loss(val_df["label"], p_va_lr))
pd.Series(base_val, name="baseline (validation)").round(4).to_frame()

## 5. Preprocessing & model
**Pipeline:** lower-case → URLs/e-mail addresses replaced by `<url>` / `<email>` → regex tokenisation → vocabulary built from the *training* set only (top 30k tokens, min. frequency 2) → ids truncated/padded to 256 tokens.

**Architecture (BiLSTM):**
`Embedding(128)` → `Dropout` → `BiLSTM(128 per direction)` → `[masked mean-pool ‖ masked max-pool]` → `Dropout` → `Linear(512→64)` + `ReLU` → `Dropout` → `Linear(64→1)` (logit, sigmoid applied in the loss / at inference).

In [ ]:
train_tokens = [tokenize(t) for t in train_df["text"]]
vocab = build_vocab(train_tokens, MAX_VOCAB, MIN_FREQ)

val_tokens = [t for toks in map(tokenize, val_df["text"]) for t in toks]
oov = np.mean([t not in vocab for t in val_tokens])
print(f"Vocabulary size: {len(vocab):,} | validation OOV token rate: {oov:.2%}")

def make_loader(frame, shuffle):
    ids, lengths = encode_batch(frame["text"].tolist(), vocab, MAX_LEN)
    y = torch.tensor(frame["label"].values, dtype=torch.float32)
    return DataLoader(TensorDataset(ids, lengths, y), batch_size=BATCH, shuffle=shuffle)

train_loader, val_loader, test_loader = make_loader(train_df, True), make_loader(val_df, False), make_loader(test_df, False)

model = PhishingBiLSTM(len(vocab), EMBED_DIM, HIDDEN, DROPOUT).to(DEVICE)
print(model)
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Training loop – per-epoch metric logging + patience-based early stopping
* After **every epoch** the loss, accuracy, precision, recall, F1 and AUC-ROC are computed for both train and validation data and appended to a table (also saved to `artifacts/training_history.csv`).
* **Early stopping:** training halts when validation loss has not improved by at least `MIN_DELTA` for `PATIENCE` consecutive epochs; the weights from the best epoch are then restored.
* `ReduceLROnPlateau` halves the learning rate when validation loss stalls (before patience runs out).
* Train metrics are running averages over the epoch with dropout active, so they are slightly pessimistic.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

class EarlyStopping:
    def __init__(self, patience, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best, self.best_epoch, self.counter, self.best_state = float("inf"), 0, 0, None
    def step(self, val_loss, model, epoch):
        if val_loss < self.best - self.min_delta:
            self.best, self.best_epoch, self.counter = val_loss, epoch, 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, n, probs, ys = 0.0, 0, [], []
    with torch.set_grad_enabled(training):
        for ids, lengths, y in loader:
            ids, lengths, y = ids.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
            logits = model(ids, lengths)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * y.size(0); n += y.size(0)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(y.cpu().numpy())
    y_true, y_prob = np.concatenate(ys).astype(int), np.concatenate(probs)
    return compute_metrics(y_true, y_prob, loss=total / n), y_true, y_prob

In [ ]:
set_seed()
stopper, history = EarlyStopping(PATIENCE, MIN_DELTA), []
t_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    tr, _, _ = run_epoch(model, train_loader, optimizer)
    va, _, _ = run_epoch(model, val_loader)
    history.append({"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()},
                    **{f"val_{k}": v for k, v in va.items()}, "lr": optimizer.param_groups[0]["lr"]})
    scheduler.step(va["loss"])
    should_stop = stopper.step(va["loss"], model, epoch)

    print(f"Epoch {epoch:02d} | train loss {tr['loss']:.4f} acc {tr['accuracy']:.4f} | "
          f"val loss {va['loss']:.4f} acc {va['accuracy']:.4f} P {va['precision']:.4f} "
          f"R {va['recall']:.4f} F1 {va['f1']:.4f} AUC {va['auc']:.4f} | "
          f"patience {stopper.counter}/{PATIENCE} | {time.time()-t0:.1f}s")
    if should_stop:
        print(f"\n>> Early stopping triggered at epoch {epoch}. Best epoch: {stopper.best_epoch} "
              f"(val loss {stopper.best:.4f}). Restoring best weights.")
        break
else:
    print("\nReached MAX_EPOCHS without triggering early stopping.")

model.load_state_dict(stopper.best_state)
print(f"Total training time: {(time.time()-t_start)/60:.1f} min")

history_df = pd.DataFrame(history)
history_df.to_csv(ARTIFACTS / "training_history.csv", index=False)
history_df.round(4)

## 7. Metric graphs (Matplotlib)
### 7.1 Training curves

In [ ]:
h, best = history_df, stopper.best_epoch
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0, 0].plot(h.epoch, h.train_loss, "o-", label="train"); ax[0, 0].plot(h.epoch, h.val_loss, "s-", label="validation")
ax[0, 0].axvline(best, ls="--", c="grey", label=f"best epoch ({best})")
ax[0, 0].set(title="Loss curve (BCE)", xlabel="epoch", ylabel="loss"); ax[0, 0].legend()

ax[0, 1].plot(h.epoch, h.train_accuracy, "o-", label="train"); ax[0, 1].plot(h.epoch, h.val_accuracy, "s-", label="validation")
ax[0, 1].axvline(best, ls="--", c="grey"); ax[0, 1].set(title="Accuracy", xlabel="epoch"); ax[0, 1].legend()

ax[1, 0].plot(h.epoch, h.val_precision, "o-", label="precision"); ax[1, 0].plot(h.epoch, h.val_recall, "s-", label="recall")
ax[1, 0].plot(h.epoch, h.val_f1, "^-", label="F1"); ax[1, 0].axvline(best, ls="--", c="grey")
ax[1, 0].set(title="Validation precision / recall / F1", xlabel="epoch"); ax[1, 0].legend()

ax[1, 1].plot(h.epoch, h.train_auc, "o-", label="train"); ax[1, 1].plot(h.epoch, h.val_auc, "s-", label="validation")
ax[1, 1].axvline(best, ls="--", c="grey"); ax[1, 1].set(title="AUC-ROC", xlabel="epoch"); ax[1, 1].legend()

plt.suptitle("Training history (dashed line = epoch whose weights were restored)", y=1.01)
plt.tight_layout(); plt.show()

## 8. Final evaluation on the held-out test set
Run **once**. Both models are scored on the same test e-mails.

In [ ]:
test_m, y_test, p_test = run_epoch(model, test_loader)
p_te_lr = lr.predict_proba(X_te)[:, 1]
base_test = compute_metrics(y_test, p_te_lr, loss=log_loss(y_test, p_te_lr))

results = pd.DataFrame({"TF-IDF + LogReg (baseline)": base_test, "BiLSTM (main model)": test_m}).round(4)
results

In [ ]:
y_pred = (p_test >= THRESHOLD).astype(int)
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"], digits=4))

### 8.1 ROC and Precision–Recall curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))

for name, p, c in [("BiLSTM", p_test, "#c0392b"), ("TF-IDF + LogReg", p_te_lr, "#2c6fbb")]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, c=c, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, p):.4f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
ax[0].set(title="ROC curve (test set)", xlabel="False positive rate", ylabel="True positive rate"); ax[0].legend(loc="lower right")

for name, p, c in [("BiLSTM", p_test, "#c0392b"), ("TF-IDF + LogReg", p_te_lr, "#2c6fbb")]:
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[1].plot(rc, pr, c=c, lw=2, label=f"{name} (AP = {average_precision_score(y_test, p):.4f})")
ax[1].axhline(y_test.mean(), c="k", ls="--", lw=1, label="no-skill")
ax[1].set(title="Precision–Recall curve (test set)", xlabel="Recall", ylabel="Precision"); ax[1].legend(loc="lower left")
plt.tight_layout(); plt.show()

### 8.2 Confusion matrix and score distribution

In [ ]:
def plot_cm(ax, cm, title, fmt):
    im = ax.imshow(cm, cmap="Blues")
    ax.set(title=title, xticks=[0, 1], yticks=[0, 1], xticklabels=["Legitimate", "Phishing"],
           yticklabels=["Legitimate", "Phishing"], xlabel="Predicted", ylabel="Actual"); ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13, fontweight="bold")

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
plot_cm(ax[0], cm, "Confusion matrix (counts)", "d")
plot_cm(ax[1], cm / cm.sum(axis=1, keepdims=True), "Confusion matrix (row-normalised)", ".3f")
ax[2].hist(p_test[y_test == 0], bins=30, alpha=0.65, label="Legitimate", color="#4c9f70")
ax[2].hist(p_test[y_test == 1], bins=30, alpha=0.65, label="Phishing", color="#c0392b")
ax[2].axvline(THRESHOLD, c="k", ls="--", label="threshold"); ax[2].set(title="Predicted phishing probability", xlabel="P(phishing)"); ax[2].legend()
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  TN={tn}  FP={fp} (false alarms)  FN={fn} (missed phishing)")
print(f"False-positive rate: {fp/(fp+tn):.2%} | False-negative rate: {fn/(fn+tp):.2%}")

### 8.3 Error analysis – what does the model get wrong?

In [ ]:
errs = test_df.assign(prob=p_test, pred=y_pred).reset_index(drop=True)
fn_df = errs[(errs.label == 1) & (errs.pred == 0)].sort_values("prob").head(4)
fp_df = errs[(errs.label == 0) & (errs.pred == 1)].sort_values("prob", ascending=False).head(4)

print("=== MISSED PHISHING (false negatives) ===")
for _, r in fn_df.iterrows(): print(f"[p={r.prob:.3f}] {r.text[:300]!r}\n")
print("=== FALSE ALARMS (false positives) ===")
for _, r in fp_df.iterrows(): print(f"[p={r.prob:.3f}] {r.text[:300]!r}\n")

## 9. Save model artefacts
These files are loaded by `api.py` (FastAPI), which the Tkinter app (`gui.py`) calls.

In [ ]:
torch.save(model.state_dict(), ARTIFACTS / "model.pt")
(ARTIFACTS / "vocab.json").write_text(json.dumps(vocab))
(ARTIFACTS / "config.json").write_text(json.dumps({
    "max_len": MAX_LEN, "embed_dim": EMBED_DIM, "hidden": HIDDEN, "dropout": DROPOUT,
    "threshold": THRESHOLD, "class_names": ["legitimate", "phishing"]}, indent=2))
(ARTIFACTS / "test_metrics.json").write_text(json.dumps(
    {"bilstm": test_m, "baseline": base_test, "best_epoch": stopper.best_epoch}, indent=2))
print("Saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

# sanity check: reload through the same class the API uses and compare with in-notebook predictions
clf = PhishingClassifier.from_dir(ARTIFACTS, device="cpu")
check = clf.predict_proba(test_df["text"].tolist()[:200])
print("Max |reloaded - in-notebook| on 200 test e-mails:", np.abs(check - p_test[:200]).max().round(6))

## 10. Explainable AI (bonus)
### 10.1 Global view – which words does the baseline associate with each class?

In [ ]:
names = np.array(tfidf.get_feature_names_out()); coef = lr.coef_[0]
top_p, top_l = np.argsort(coef)[-15:], np.argsort(coef)[:15]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].barh(names[top_p], coef[top_p], color="#c0392b"); ax[0].set_title("Strongest PHISHING indicators (LogReg)")
ax[1].barh(names[top_l], -coef[top_l], color="#4c9f70"); ax[1].set_title("Strongest LEGITIMATE indicators (LogReg)")
plt.tight_layout(); plt.show()

### 10.2 Local view – LIME explanation of individual BiLSTM predictions
LIME perturbs an e-mail by removing words and measures how the predicted probability changes. **Red** bars push the prediction towards phishing, **green** bars towards legitimate. The same explanation is exposed through the API's `/explain` endpoint and highlighted in the desktop app.

In [ ]:
pool = test_df.reset_index(drop=True).assign(prob=p_test, pred=y_pred)
picks = {
    "Correctly flagged PHISHING":  pool[(pool.label == 1) & (pool.pred == 1)].sort_values("prob", ascending=False).iloc[0],
    "Correctly passed LEGITIMATE": pool[(pool.label == 0) & (pool.pred == 0)].sort_values("prob").iloc[0],
}
if ((pool.label == 1) & (pool.pred == 0)).any():
    picks["MISSED phishing (false negative)"] = pool[(pool.label == 1) & (pool.pred == 0)].sort_values("prob").iloc[0]

for title, row in picks.items():
    exp = clf.lime_explanation(row.text, num_features=10, num_samples=1000)
    fig = exp.as_pyplot_figure(); fig.set_size_inches(7, 3.4)
    plt.title(f"{title}  (P(phishing) = {row.prob:.3f})"); plt.tight_layout(); plt.show()
    print(row.text[:350].replace("\n", " "), "...\n")

## 11. Numbers for the theory write-up
Copy these into Sections 3.2 and 3.3.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Dataset after cleaning : {len(df):,} e-mails ({df.label.mean():.1%} phishing)")
print(f"Split                  : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"Vocabulary             : {len(vocab):,} tokens | max sequence length {MAX_LEN}")
print(f"Model parameters       : {n_params:,}")
print(f"Best epoch / stopped at: {stopper.best_epoch} / {len(history_df)} (patience = {PATIENCE})")
print(f"Test F1 BiLSTM vs base : {test_m['f1']:.4f} vs {base_test['f1']:.4f}")
print(f"Test AUC BiLSTM vs base: {test_m['auc']:.4f} vs {base_test['auc']:.4f}")
print(f"Missed phishing (FN)   : {fn} of {fn+tp} ({fn/(fn+tp):.2%}) | false alarms (FP): {fp} of {fp+tn} ({fp/(fp+tn):.2%})")

## 12. Run the desktop application
```bash
uvicorn api:app          # terminal 1 – model API  (interactive docs: http://127.0.0.1:8000/docs)
python gui.py            # terminal 2 – Tkinter app
```